# Performance Review Validation

This notebook validates annual performance reviews for the complete synthetic workforce.

The main rules are:

- Review IDs are complete and unique.
- Employees and reviewers must exist.
- Reviewers must be active managers.
- Employees cannot review themselves.
- Employees and reviewers must belong to the same department.
- Reviews cannot occur before hire or after employment ends.
- Reviews occur after at least one complete year of employment.
- Ratings must remain between 1.0 and 5.0.
- Goal completion must remain between 0% and 120%.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

departments = pd.read_csv(
    RAW_DATA_DIR / "departments.csv"
)

performance_reviews = pd.read_csv(
    RAW_DATA_DIR / "performance_reviews.csv",
    parse_dates=["review_date"],
)

employees["manager_id"] = (
    employees["manager_id"]
    .astype("Int64")
)

performance_reviews[
    "promotion_recommended"
] = (
    performance_reviews[
        "promotion_recommended"
    ]
    .astype("boolean")
)

print("Employees:", employees.shape)
print(
    "Performance reviews:",
    performance_reviews.shape,
)

Employees: (10000, 15)
Performance reviews: (17972, 8)


## 1. Initial inspection

In [2]:
performance_reviews.head(10)

,review_id,employee_id,review_date,review_period,performance_rating,goal_completion,promotion_recommended,reviewer_id
0,700001,100002,2023-04-03,2023 Annual,3.7,94.0,False,100001
1,700002,100002,2024-04-03,2024 Annual,3.9,91.0,True,100001
2,700003,100002,2025-04-03,2025 Annual,3.3,87.7,False,100001
3,700004,100002,2026-04-03,2026 Annual,3.8,94.6,True,100001
4,700005,100003,2023-06-28,2023 Annual,3.6,86.3,False,100001
5,700006,100003,2024-06-28,2024 Annual,3.4,83.8,True,100001
6,700007,100003,2025-06-28,2025 Annual,3.9,87.4,False,100001
7,700008,100003,2026-06-28,2026 Annual,4.2,116.4,False,100001
8,700009,100004,2023-03-24,2023 Annual,4.0,104.5,False,100001
9,700010,100004,2024-03-24,2024 Annual,3.7,93.7,False,100001


In [3]:
performance_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 17972 entries, 0 to 17971
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   review_id              17972 non-null  int64         
 1   employee_id            17972 non-null  int64         
 2   review_date            17972 non-null  datetime64[us]
 3   review_period          17972 non-null  str           
 4   performance_rating     17972 non-null  float64       
 5   goal_completion        17972 non-null  float64       
 6   promotion_recommended  17972 non-null  boolean       
 7   reviewer_id            17972 non-null  int64         
dtypes: boolean(1), datetime64[us](1), float64(2), int64(3), str(1)
memory usage: 1018.1 KB


## 2. Employee review coverage

In [4]:
reviews_per_employee = (
    performance_reviews
    .groupby("employee_id")
    .size()
    .rename("review_count")
)

reviews_per_employee.describe()

count    7639.000000
mean        2.352664
std         1.099759
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         5.000000
Name: review_count, dtype: float64

In [5]:
print(
    "Employees with reviews:",
    reviews_per_employee.size,
)

print(
    "Employees without reviews:",
    len(employees)
    - reviews_per_employee.size,
)

print(
    "Minimum reviews:",
    reviews_per_employee.min(),
)

print(
    "Maximum reviews:",
    reviews_per_employee.max(),
)

Employees with reviews: 7639
Employees without reviews: 2361
Minimum reviews: 1
Maximum reviews: 5


## 3. Employee and reviewer relationships

In [6]:
employee_details = (
    employees[
        [
            "employee_id",
            "hire_date",
            "termination_date",
            "department_id",
            "manager_id",
            "organizational_level",
        ]
    ]
    .rename(
        columns={
            "hire_date": "employee_hire_date",
            "termination_date": (
                "employee_termination_date"
            ),
            "department_id": (
                "employee_department_id"
            ),
            "manager_id": (
                "expected_reviewer_id"
            ),
            "organizational_level": (
                "employee_level"
            ),
        }
    )
)

reviewer_details = (
    employees[
        [
            "employee_id",
            "hire_date",
            "department_id",
            "employment_status",
            "organizational_level",
        ]
    ]
    .rename(
        columns={
            "employee_id": "reviewer_id",
            "hire_date": "reviewer_hire_date",
            "department_id": (
                "reviewer_department_id"
            ),
            "employment_status": (
                "reviewer_status"
            ),
            "organizational_level": (
                "reviewer_level"
            ),
        }
    )
)

review_details = (
    performance_reviews
    .merge(
        employee_details,
        on="employee_id",
        how="left",
    )
    .merge(
        reviewer_details,
        on="reviewer_id",
        how="left",
    )
)

review_details.head()

,review_id,employee_id,review_date,review_period,performance_rating,goal_completion,promotion_recommended,reviewer_id,employee_hire_date,employee_termination_date,employee_department_id,expected_reviewer_id,employee_level,reviewer_hire_date,reviewer_department_id,reviewer_status,reviewer_level
0,700001,100002,2023-04-03,2023 Annual,3.7,94.0,False,100001,2022-04-03,NaT,1,100001,Senior Manager,2022-10-17,1,Active,Department Head
1,700002,100002,2024-04-03,2024 Annual,3.9,91.0,True,100001,2022-04-03,NaT,1,100001,Senior Manager,2022-10-17,1,Active,Department Head
2,700003,100002,2025-04-03,2025 Annual,3.3,87.7,False,100001,2022-04-03,NaT,1,100001,Senior Manager,2022-10-17,1,Active,Department Head
3,700004,100002,2026-04-03,2026 Annual,3.8,94.6,True,100001,2022-04-03,NaT,1,100001,Senior Manager,2022-10-17,1,Active,Department Head
4,700005,100003,2023-06-28,2023 Annual,3.6,86.3,False,100001,2021-06-28,NaT,1,100001,Senior Manager,2022-10-17,1,Active,Department Head


## 4. Review-date checks

In [7]:
review_details[
    "employment_end_date"
] = (
    review_details[
        "employee_termination_date"
    ]
    .fillna(
        pd.Timestamp("2026-06-30")
    )
)

review_details[
    "first_anniversary"
] = (
    review_details[
        "employee_hire_date"
    ]
    + pd.DateOffset(years=1)
)

date_checks = pd.Series(
    {
        "no review occurs before hire": (
            review_details[
                "review_date"
            ]
            .ge(
                review_details[
                    "employee_hire_date"
                ]
            )
            .all()
        ),
        "no review occurs after employment": (
            review_details[
                "review_date"
            ]
            .le(
                review_details[
                    "employment_end_date"
                ]
            )
            .all()
        ),
        "reviews occur after one year": (
            review_details[
                "review_date"
            ]
            .ge(
                review_details[
                    "first_anniversary"
                ]
            )
            .all()
        ),
        "reviewer was already employed": (
            review_details[
                "review_date"
            ]
            .ge(
                review_details[
                    "reviewer_hire_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

date_checks

no review occurs before hire         True
no review occurs after employment    True
reviews occur after one year         True
reviewer was already employed        True
Name: passed, dtype: bool

## 5. Reviewer checks

In [8]:
manager_levels = {
    "Department Head",
    "Senior Manager",
    "Team Manager",
}

reviewer_checks = pd.Series(
    {
        "reviewers are valid employees": (
            review_details[
                "reviewer_hire_date"
            ].notna().all()
        ),
        "employees do not review themselves": (
            review_details[
                "employee_id"
            ]
            .ne(
                review_details[
                    "reviewer_id"
                ]
            )
            .all()
        ),
        "reviewer matches current manager": (
            review_details[
                "reviewer_id"
            ]
            .eq(
                review_details[
                    "expected_reviewer_id"
                ]
            )
            .all()
        ),
        "employee and reviewer share department": (
            review_details[
                "employee_department_id"
            ]
            .eq(
                review_details[
                    "reviewer_department_id"
                ]
            )
            .all()
        ),
        "all reviewers are active": (
            review_details[
                "reviewer_status"
            ].eq("Active").all()
        ),
        "all reviewers are managers": (
            review_details[
                "reviewer_level"
            ]
            .isin(manager_levels)
            .all()
        ),
    },
    name="passed",
)

reviewer_checks

reviewers are valid employees             True
employees do not review themselves        True
reviewer matches current manager          True
employee and reviewer share department    True
all reviewers are active                  True
all reviewers are managers                True
Name: passed, dtype: bool

## 6. Review-period checks

In [9]:
expected_review_period = (
    performance_reviews[
        "review_date"
    ]
    .dt.year
    .astype(str)
    + " Annual"
)

period_checks = pd.Series(
    {
        "period matches review year": (
            performance_reviews[
                "review_period"
            ]
            .eq(
                expected_review_period
            )
            .all()
        ),
        "employee-period combinations are unique": (
            not performance_reviews
            .duplicated(
                subset=[
                    "employee_id",
                    "review_period",
                ]
            )
            .any()
        ),
    },
    name="passed",
)

period_checks

period matches review year                 True
employee-period combinations are unique    True
Name: passed, dtype: bool

## 7. Score checks

In [10]:
score_checks = pd.Series(
    {
        "ratings are between 1 and 5": (
            performance_reviews[
                "performance_rating"
            ]
            .between(
                1.0,
                5.0,
            )
            .all()
        ),
        "goal completion is between 0 and 120": (
            performance_reviews[
                "goal_completion"
            ]
            .between(
                0,
                120,
            )
            .all()
        ),
        "promotion recommendation is boolean": (
            performance_reviews[
                "promotion_recommended"
            ]
            .isin(
                [
                    True,
                    False,
                ]
            )
            .all()
        ),
    },
    name="passed",
)

score_checks

ratings are between 1 and 5             True
goal completion is between 0 and 120    True
promotion recommendation is boolean     True
Name: passed, dtype: bool

In [11]:
department_head_ids = set(
    employees.loc[
        employees[
            "organizational_level"
        ]
        == "Department Head",
        "employee_id",
    ]
)

reviewed_employee_ids = set(
    performance_reviews[
        "employee_id"
    ]
)

department_head_review_count = len(
    department_head_ids.intersection(
        reviewed_employee_ids
    )
)

print(
    "Department heads with reviews:",
    department_head_review_count,
)

Department heads with reviews: 0


## 8. Performance summary

In [12]:
performance_reviews[
    [
        "performance_rating",
        "goal_completion",
    ]
].describe().round(2)

,performance_rating,goal_completion
count,17972.00,17972.00
mean,3.50,91.99
std,0.52,10.11
min,1.10,47.00
25%,3.10,85.10
50%,3.50,92.00
75%,3.90,98.90
max,5.00,120.00


In [13]:
rating_distribution = (
    performance_reviews[
        "performance_rating"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "performance_rating"
    )
    .reset_index(
        name="review_count"
    )
)

rating_distribution

,performance_rating,review_count
0,1.1,1
1,1.3,1
2,1.5,3
3,1.6,5
4,1.7,9
5,1.8,12
6,1.9,12
7,2.0,26
8,2.1,40
9,2.2,66


In [14]:
performance_by_level = (
    review_details
    .groupby("employee_level")
    .agg(
        review_count=(
            "review_id",
            "count",
        ),
        average_rating=(
            "performance_rating",
            "mean",
        ),
        average_goal_completion=(
            "goal_completion",
            "mean",
        ),
        promotion_recommendation_rate=(
            "promotion_recommended",
            "mean",
        ),
    )
    .round(3)
)

performance_by_level[
    "promotion_recommendation_rate"
] = (
    performance_by_level[
        "promotion_recommendation_rate"
    ]
    * 100
).round(2)

performance_by_level

,review_count,average_rating,average_goal_completion,promotion_recommendation_rate
employee_level,,,,
Individual Contributor,15594,3.480,91.786,9.9
Senior Manager,233,3.617,93.409,12.4
Team Manager,2145,3.593,93.331,11.7


In [15]:
performance_by_department = (
    review_details
    .groupby(
        "employee_department_id"
    )
    .agg(
        review_count=(
            "review_id",
            "count",
        ),
        employees_reviewed=(
            "employee_id",
            "nunique",
        ),
        average_rating=(
            "performance_rating",
            "mean",
        ),
        average_goal_completion=(
            "goal_completion",
            "mean",
        ),
    )
    .reset_index()
    .merge(
        departments,
        left_on=(
            "employee_department_id"
        ),
        right_on="department_id",
        how="left",
    )
)

performance_by_department[
    [
        "department_name",
        "review_count",
        "employees_reviewed",
        "average_rating",
        "average_goal_completion",
    ]
].round(2)

,department_name,review_count,employees_reviewed,average_rating,average_goal_completion
0,Engineering,3667,1559,3.59,93.17
1,Manufacturing,4376,1883,3.42,90.92
2,Supply Chain,2215,931,3.51,92.21
3,Sales,1722,749,3.53,92.50
4,Finance,1416,591,3.52,92.34
5,Human Resources,1351,557,3.54,92.63
6,Information Technology,1828,774,3.51,92.02
7,Customer Support,1397,595,3.36,90.28


## 9. Complete validation summary

In [16]:
basic_checks = pd.Series(
    {
        "table has eight columns": (
            len(
                performance_reviews.columns
            )
            == 8
        ),
        "review IDs are complete": (
            performance_reviews[
                "review_id"
            ].notna().all()
        ),
        "review IDs are unique": (
            performance_reviews[
                "review_id"
            ].is_unique
        ),
        "employee IDs are valid": (
            set(
                performance_reviews[
                    "employee_id"
                ]
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "reviewer IDs are valid": (
            set(
                performance_reviews[
                    "reviewer_id"
                ]
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "department heads have no reviews": (
            department_head_review_count
            == 0
        ),
    },
    name="passed",
)

all_checks = pd.concat(
    [
        basic_checks,
        date_checks,
        reviewer_checks,
        period_checks,
        score_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,table has eight columns,True
1,review IDs are complete,True
2,review IDs are unique,True
3,employee IDs are valid,True
4,reviewer IDs are valid,True
5,department heads have no reviews,True
6,no review occurs before hire,True
7,no review occurs after employment,True
8,reviews occur after one year,True
9,reviewer was already employed,True


In [17]:
if validation_results["passed"].all():
    print(
        "All performance-review "
        "validation checks passed."
    )
else:
    print(
        "One or more performance-review "
        "validation checks failed."
    )

All performance-review validation checks passed.


## 10. Conclusions

The synthetic performance-review table successfully represents annual reviews across the workforce.

### Successful checks

- Review IDs are complete and unique.
- Employees and reviewers are valid employee records.
- Employees do not review themselves.
- Reviewers are active managers.
- Employees and reviewers belong to the same department.
- Reviews do not occur before hire or after employment ends.
- Reviews occur after at least one complete year of employment.
- Performance ratings remain between 1.0 and 5.0.
- Goal completion remains between 0% and 120%.
- Employee-period combinations are unique.
- All performance-review validation checks passed.

### Current simplifications

- Department heads do not receive reviews because executives are not yet represented.
- Historical reviews use the employee's current manager.
- Reviews before the current manager's hire date are skipped.
- Review dates occur on employment anniversaries.
- Performance ratings and goal completion are synthetic.
- Promotion recommendations do not yet create promotion events automatically.